# Hybrid PPO Baseline

Notebook version of `src/ppo/elurant_ppo.py`.

This is the discrete-lane plus continuous-throttle PPO baseline, with local outputs under `artifacts/ppo/hybrid_ppo_baseline`.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

from stable_baselines3 import PPO
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor


def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "src").exists() and (candidate / "notebooks").exists():
            return candidate
        nested = candidate / "highway-rl-decision-making"
        if (nested / "src").exists() and (nested / "notebooks").exists():
            return nested
    raise RuntimeError("Could not locate the project root from the current working directory.")


PROJECT_ROOT = find_project_root()
NOTEBOOK_DIR = PROJECT_ROOT / "notebooks" / "ppo"
RESULTS_ROOT = PROJECT_ROOT / "artifacts" / "ppo" / "hybrid_ppo_baseline"
SCRIPT_DIR = PROJECT_ROOT / "src" / "deep_learning" / "Elurant_PPO"
if str(SCRIPT_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPT_DIR))

import elurant_ppo

RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
TB_DIR = RESULTS_ROOT / "tensorboard"
TB_DIR.mkdir(parents=True, exist_ok=True)
print("Notebook dir:", NOTEBOOK_DIR)
print("Script dir:", SCRIPT_DIR)
print("Results dir:", RESULTS_ROOT)


## Config

In [ ]:
train_cfg = {
    'timesteps': 10000,
    'eval_episodes': 5,
    'n_steps': 512,
    'batch_size': 64,
    'monitor_every_steps': 500,
    'learning_rate': 3e-4,
    'gamma': 0.99,
    'gae_lambda': 0.95,
    'clip_range': 0.2,
    'ent_coef': 0.01,
    'seed': 42,
    'device': 'auto',
}

wrapper_kwargs = {
    'lane_action_mode': 'intent',
    'lane_safety_checks': True,
    'throttle_safety_checks': True,
}

print(json.dumps(train_cfg, indent=2))
print(json.dumps(wrapper_kwargs, indent=2))


## Train And Evaluate

In [ ]:
env = elurant_ppo.make_env(render_mode='rgb_array', **wrapper_kwargs)
env.reset(seed=train_cfg['seed'])

model = PPO(
    policy='MlpPolicy',
    env=env,
    learning_rate=train_cfg['learning_rate'],
    n_steps=train_cfg['n_steps'],
    batch_size=train_cfg['batch_size'],
    n_epochs=10,
    gamma=train_cfg['gamma'],
    gae_lambda=train_cfg['gae_lambda'],
    clip_range=train_cfg['clip_range'],
    ent_coef=train_cfg['ent_coef'],
    verbose=1,
    tensorboard_log=str(TB_DIR),
    device=train_cfg['device'],
    seed=train_cfg['seed'],
)

monitor_cb = elurant_ppo.TimestepMonitorCallback(every_n_steps=train_cfg['monitor_every_steps'])
model.learn(total_timesteps=train_cfg['timesteps'], callback=monitor_cb, progress_bar=False)

model_path = RESULTS_ROOT / 'elurant_ppo.zip'
model.save(str(model_path.with_suffix('')))

eval_env = Monitor(elurant_ppo.make_env(render_mode='rgb_array', **wrapper_kwargs))
mean_reward, std_reward = evaluate_policy(model, eval_env, n_eval_episodes=train_cfg['eval_episodes'])
summary = {
    'mean_reward': float(mean_reward),
    'std_reward': float(std_reward),
    'model_path': str(model_path),
    'tensorboard_dir': str(TB_DIR),
}
(RESULTS_ROOT / 'summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
print(json.dumps(summary, indent=2))

eval_env.close()
env.close()
